[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C19_Bayesian_ML_Course/05_graphical_models/05_graphical_models.ipynb)

# 05 · 概率图模型 PGM（纯 numpy 从零）

目标：从零实现 **因子运算**（乘积、边缘化）、**变量消去**、**信念传播（sum-product）**、**d-分离**，全部与**暴力枚举联合分布**对拍。

路线：因子表示与乘积/边缘 → 构造贝叶斯网（草地湿）→ 暴力边缘（参照）→ 变量消去（对拍暴力）→ 带证据的条件推断 → 因子图上的 BP（对拍暴力）→ d-分离判定（对拍数值条件独立）→ ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：**图结构 ⟺ 条件独立 ⟺ 因子分解**。推断 = 因子的『乘积』与『求和消去』；精确推断的难度由**树宽**（而非变量数）决定。一切都对拍暴力枚举验证。

## 1 · 因子：带命名变量轴的多维数组

因子 = 一个非负函数 $f(x_a)$，用 numpy 数组表示，每个轴对应一个变量。两个核心操作：**因子乘积**（对齐共享变量后逐元素相乘）与**边缘化**（对某变量求和消去）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def check_allclose(name, got, ref, atol=1e-8, rtol=1e-5):
    got=np.asarray(got,float); ref=np.asarray(ref,float)
    ok=np.allclose(got,ref,atol=atol,rtol=rtol)
    err=float(np.max(np.abs(got-ref))) if got.size else 0.0
    print(f'[{name:<34}] allclose={ok}  max|err|={err:.2e}')
    assert ok, f'{name} 不一致'; return ok

class Factor:
    '''离散因子：vars=变量名元组, table=numpy 数组(各轴对应一个变量, 取值数=该轴长度)。'''
    def __init__(self, vars, table):
        self.vars = tuple(vars)
        self.table = np.asarray(table, float)
        assert self.table.ndim == len(self.vars)
    def __repr__(self):
        return f'Factor({self.vars}, shape={self.table.shape})'

def factor_product(f1, f2):
    '''两因子乘积：对齐到变量并集，广播相乘。'''
    vars_out = list(f1.vars) + [v for v in f2.vars if v not in f1.vars]
    # 把每个因子 reshape/broadcast 到 vars_out 的形状
    def expand(f):
        shape = [f.table.shape[f.vars.index(v)] if v in f.vars else 1 for v in vars_out]
        # 先把已有轴按 vars_out 顺序排列
        perm = [f.vars.index(v) for v in vars_out if v in f.vars]
        t = np.transpose(f.table, perm)
        return t.reshape(shape)
    return Factor(vars_out, expand(f1) * expand(f2))

def factor_marginalize(f, var):
    '''对 var 求和消去。'''
    ax = f.vars.index(var)
    new_vars = tuple(v for v in f.vars if v != var)
    return Factor(new_vars, f.table.sum(axis=ax))

# 测试：两个简单因子
fA = Factor(('A',), [0.6, 0.4])
fAB = Factor(('A','B'), [[0.7,0.3],[0.2,0.8]])   # p(B|A)
prod = factor_product(fA, fAB)                    # p(A)p(B|A)=p(A,B)
assert set(prod.vars) == {'A','B'}
# 边缘出 p(B): sum_A p(A,B)
fB = factor_marginalize(prod, 'A')
# 手算: p(B=0)=0.6*0.7+0.4*0.2=0.5, p(B=1)=0.6*0.3+0.4*0.8=0.5
check_allclose('p(B) via factor ops', fB.table, [0.5, 0.5])
# 联合应归一化
check_allclose('p(A,B) sums to 1', prod.table.sum(), 1.0)
print('✅ 因子乘积 + 边缘化：图推断的两个原子操作')

## 2 · 构造贝叶斯网（草地湿）+ 暴力枚举参照

经典网络：雨 $R$、洒水器 $S$、草湿 $W$，结构 $R\to S$, $R\to W$, $S\to W$。
$p(R,S,W)=p(R)p(S|R)p(W|R,S)$。先**暴力**算出完整联合表，作为后续所有推断的**绝对参照**。

In [ ]:
# CPTs（变量取值 0/1）
p_R = Factor(('R',), [0.8, 0.2])                  # p(R): 不雨0.8, 雨0.2
p_S_R = Factor(('R','S'), [[0.6, 0.4],            # p(S|R=0): 不开0.6,开0.4
                           [0.99, 0.01]])         # p(S|R=1): 下雨基本不开
# p(W|R,S): 轴顺序 (R,S,W)
p_W_RS = Factor(('R','S','W'), np.array([
    [[1.0, 0.0],    # R=0,S=0: 草干
     [0.1, 0.9]],   # R=0,S=1: 洒水->多半湿
    [[0.2, 0.8],    # R=1,S=0: 下雨->多半湿
     [0.01,0.99]]   # R=1,S=1: 几乎必湿
]))

# 暴力联合 p(R,S,W)
joint = factor_product(factor_product(p_R, p_S_R), p_W_RS)
assert set(joint.vars) == {'R','S','W'}
check_allclose('联合 p(R,S,W) 归一化', joint.table.sum(), 1.0)

def brute_marginal(joint, target):
    '''暴力：对联合中除 target 外所有变量求和。'''
    f = joint
    for v in joint.vars:
        if v != target:
            f = factor_marginalize(f, v)
    return f

pW_brute = brute_marginal(joint, 'W')
print('暴力 p(W) =', np.round(pW_brute.table, 4))
check_allclose('p(W) 归一化', pW_brute.table.sum(), 1.0)
print('✅ 贝叶斯网联合 + 暴力边缘（这是后续所有推断的 ground truth）')

## 3 · 变量消去：对拍暴力边缘

变量消去 = 按顺序逐个消去非目标变量：取出含该变量的因子→相乘→求和消去→放回。应得到与暴力**完全相同**的边缘，但不显式构造完整联合（在大网络上高效得多）。

In [ ]:
def variable_elimination(factors, target, elim_order):
    '''变量消去求 p(target)。factors: Factor 列表; elim_order: 要消去的变量顺序。'''
    Phi = list(factors)
    for z in elim_order:
        # 取出所有含 z 的因子
        with_z = [f for f in Phi if z in f.vars]
        Phi = [f for f in Phi if z not in f.vars]
        if not with_z:
            continue
        # 相乘
        prod = with_z[0]
        for f in with_z[1:]:
            prod = factor_product(prod, f)
        # 对 z 求和消去
        Phi.append(factor_marginalize(prod, z))
    # 剩余因子相乘
    result = Phi[0]
    for f in Phi[1:]:
        result = factor_product(result, f)
    # 归一化
    result.table /= result.table.sum()
    return result

factors = [p_R, p_S_R, p_W_RS]
# 求 p(W): 消去 R, S
pW_ve = variable_elimination(factors, 'W', elim_order=['R', 'S'])
check_allclose('VE p(W) vs 暴力', pW_ve.table, pW_brute.table, atol=1e-10)
# 求 p(S): 消去 R, W
pS_ve = variable_elimination(factors, 'S', elim_order=['R', 'W'])
pS_brute = brute_marginal(joint, 'S')
check_allclose('VE p(S) vs 暴力', pS_ve.table, pS_brute.table, atol=1e-10)
# 消去顺序不影响结果（只影响效率）
pW_ve2 = variable_elimination(factors, 'W', elim_order=['S', 'R'])
check_allclose('VE 不同消去顺序同结果', pW_ve.table, pW_ve2.table, atol=1e-10)
print('✅ 变量消去 = 暴力边缘（且不构造完整联合，顺序不改结果只改效率）')

## 4 · 带证据的条件推断 + explaining away（解释消除）

给定证据（如观测草湿 $W=1$），求条件分布 = 把证据变量『钉』在观测值（切片因子），再做变量消去、归一化。

用它复现 **explaining away**：$p(S=1|W=1)$ vs $p(S=1|W=1,R=1)$——已知下雨后，洒水器的后验概率应**下降**。

In [ ]:
def reduce_evidence(factor, evidence):
    '''把因子中的证据变量切片到观测值。evidence: {var: value}。'''
    vars_out, table = list(factor.vars), factor.table
    for v, val in evidence.items():
        if v in vars_out:
            ax = vars_out.index(v)
            table = np.take(table, val, axis=ax)
            vars_out.pop(ax)
    return Factor(vars_out, table)

def query(factors, target, evidence, elim_order):
    '''p(target | evidence)。'''
    reduced = [reduce_evidence(f, evidence) for f in factors]
    order = [z for z in elim_order if z != target and z not in evidence]
    return variable_elimination(reduced, target, order)

# p(S | W=1)
pS_given_W = query(factors, 'S', {'W': 1}, elim_order=['R','S','W'])
# p(S | W=1, R=1)
pS_given_WR = query(factors, 'S', {'W': 1, 'R': 1}, elim_order=['R','S','W'])
print(f'p(S=1 | W=1)      = {pS_given_W.table[1]:.4f}')
print(f'p(S=1 | W=1, R=1) = {pS_given_WR.table[1]:.4f}  <- 知道下雨后，洒水器概率下降')
assert pS_given_WR.table[1] < pS_given_W.table[1], 'explaining away: 雨解释了草湿 -> 洒水器后验下降'

# 对拍暴力: 在联合上切片+归一化
def brute_conditional(joint, target, evidence):
    f = reduce_evidence(joint, evidence)
    for v in list(f.vars):
        if v != target: f = factor_marginalize(f, v)
    f.table = f.table / f.table.sum()
    return f
check_allclose('p(S|W=1) VE vs 暴力', pS_given_W.table,
               brute_conditional(joint,'S',{'W':1}).table, atol=1e-10)
print('✅ 条件推断对拍暴力；并复现 explaining away（对撞点观测后打开路径）')

## 5 · 信念传播（sum-product）在链上：一次算出所有边缘

链式马尔可夫网 $X_1 - X_2 - X_3 - X_4$（因子 = 相邻势 $\psi_{i,i+1}$）。BP 一次前向 + 一次后向传播，**精确**算出所有变量的边缘（树上精确）。对拍暴力枚举。

In [ ]:
def chain_bp_marginals(node_pots, edge_pots):
    '''链 X_0..X_{n-1} 的 sum-product BP。
       node_pots[i]: (k,) 单点势; edge_pots[i]: (k,k) 边 (i,i+1) 势。
       返回每个节点的归一化边缘。'''
    n = len(node_pots); k = len(node_pots[0])
    # 前向消息 alpha[i]: 从左传到节点 i 的消息
    fwd = [None]*n; fwd[0] = node_pots[0].copy()
    for i in range(1, n):
        msg = (fwd[i-1][:,None] * edge_pots[i-1]).sum(axis=0)  # sum_{x_{i-1}} ...
        fwd[i] = node_pots[i] * msg
    # 后向消息 beta[i]: 从右传到节点 i
    bwd = [None]*n; bwd[n-1] = np.ones(k)
    for i in range(n-2, -1, -1):
        msg = (edge_pots[i] * (node_pots[i+1]*bwd[i+1])[None,:]).sum(axis=1)
        bwd[i] = msg
    # 边缘 = 前向 * 后向 (节点势已含在 fwd)
    marg = []
    for i in range(n):
        b = fwd[i]*bwd[i]; marg.append(b/b.sum())
    return marg

# 随机构造一条链
k = 3; n = 4
node_pots = [rng.uniform(0.5, 2.0, k) for _ in range(n)]
edge_pots = [rng.uniform(0.5, 2.0, (k, k)) for _ in range(n-1)]
bp_marg = chain_bp_marginals(node_pots, edge_pots)

# 暴力：构造完整联合 (k^n) 求边缘
from itertools import product
def brute_chain_marginals(node_pots, edge_pots):
    n=len(node_pots); k=len(node_pots[0]); joint=np.zeros([k]*n)
    for idx in product(range(k), repeat=n):
        val=1.0
        for i in range(n): val*=node_pots[i][idx[i]]
        for i in range(n-1): val*=edge_pots[i][idx[i],idx[i+1]]
        joint[idx]=val
    joint/=joint.sum()
    return [joint.sum(axis=tuple(j for j in range(n) if j!=i)) for i in range(n)]
brute_marg = brute_chain_marginals(node_pots, edge_pots)
for i in range(n):
    check_allclose(f'BP 边缘 X{i} vs 暴力', bp_marg[i], brute_marg[i], atol=1e-10)
print('✅ 信念传播在树/链上精确：一次前后向算出所有边缘 = 暴力枚举')

## 6 · d-分离：纯拓扑读出条件独立（对拍数值）

d-分离用图结构判定 $X\perp Y\mid Z$，**不碰任何数值**。实现它（基于『活跃路径』搜索），并在草地湿网络上对拍数值条件独立（用因子算 $p(X,Y|Z)$ vs $p(X|Z)p(Y|Z)$）。

In [ ]:
def d_separated(edges, X, Y, Z):
    '''有向图 edges=[(parent,child),...]。判定 X ⟂ Y | Z（Z 为已观测集合）。
       用 Bayes-Ball / 活跃路径 BFS：从 X 出发能否到达 Y。'''
    Z = set(Z)
    parents = {}; children = {}
    nodes = set()
    for u, v in edges:
        children.setdefault(u, []).append(v)
        parents.setdefault(v, []).append(u)
        nodes.add(u); nodes.add(v)
    # 祖先(含自身)落在 Z 中的节点（用于对撞判断）
    def ancestors(s):
        seen=set(); stack=list(s)
        while stack:
            x=stack.pop()
            for pa in parents.get(x, []):
                if pa not in seen: seen.add(pa); stack.append(pa)
        return seen
    Z_anc = set(Z) | ancestors(Z)
    # 状态: (node, direction) direction='up'(从子来) / 'down'(从父来)
    from collections import deque
    visited=set(); q=deque()
    for d in ('up','down'): q.append((X, d))
    while q:
        node, d = q.popleft()
        if (node,d) in visited: continue
        visited.add((node,d))
        if node == Y: return False        # 找到活跃路径 -> 不独立
        if d == 'up':                      # 消息从子节点上行到 node
            if node not in Z:
                for pa in parents.get(node, []): q.append((pa,'up'))
                for ch in children.get(node, []): q.append((ch,'down'))
        else:                              # 消息从父节点下行到 node
            if node not in Z:
                for ch in children.get(node, []): q.append((ch,'down'))
            if node in Z_anc:              # 对撞点被(后代)观测 -> 打开
                for pa in parents.get(node, []): q.append((pa,'up'))
    return True

edges = [('R','S'), ('R','W'), ('S','W')]
# R 与 S: 不给证据时 R->S 直接相连 -> 不独立
assert d_separated(edges, 'R', 'S', set()) == False
# 对撞 R->W<-S: 不观测 W 时... 但 R->S 也存在，所以仍相关。换个纯对撞例子：
edges2 = [('R','W'), ('S','W')]   # 纯对撞 R->W<-S
assert d_separated(edges2, 'R', 'S', set()) == True, '纯对撞未观测W: R⟂S'
assert d_separated(edges2, 'R', 'S', {'W'}) == False, '观测W打开对撞: R,S 相关(explaining away)'
print('✅ d-分离: 纯对撞未观测则独立、观测共同后代则打开（与 explaining away 一致）')

# 对拍数值: 在纯对撞网络上验证 R⟂S（无证据）
pR = Factor(('R',),[0.7,0.3]); pS = Factor(('S',),[0.6,0.4])
pW2 = Factor(('R','S','W'), p_W_RS.table)
j2 = factor_product(factor_product(pR,pS), pW2)
# p(R,S) 应 = p(R)p(S)（边缘掉 W 后）
pRS = factor_marginalize(j2, 'W')
outer = pR.table[:,None]*pS.table[None,:]
check_allclose('纯对撞: p(R,S)=p(R)p(S)', pRS.table, outer, atol=1e-10)
print('✅ d-分离的拓扑判断与数值条件独立一致')

---
## ✏️ 练习 1：因子边缘化

实现 `factor_marginalize(f, var)`：对因子的某个变量轴求和消去，返回新因子（变量减一个）。这是变量消去和 BP 的基础操作。

In [ ]:
def factor_marginalize(f, var):
    # TODO: 找到 var 的轴, 对该轴 sum, 返回 Factor(去掉var的vars, 求和后的table)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
f = Factor(('A','B'), [[0.1,0.2,0.3],[0.15,0.05,0.2]])   # 2x3
fA = factor_marginalize(f, 'B')   # sum over B -> p over A
assert fA.vars == ('A',)
assert np.allclose(fA.table, [0.6, 0.4]), 'sum over B'
fB = factor_marginalize(f, 'A')   # sum over A -> p over B
assert fB.vars == ('B',)
assert np.allclose(fB.table, [0.25, 0.25, 0.5])
# 全部边缘掉 = 总和
assert abs(factor_marginalize(fA,'A').table - 1.0) < 1e-12
print('✅ 练习 1 通过：因子边缘化（求和消去一个变量）')

## ✏️ 练习 2：变量消去求边缘

实现 `variable_elimination(factors, target, elim_order)`：逐个消去 `elim_order` 中的变量（取含该变量的因子→相乘→边缘化→放回），最后归一化得 `p(target)`。对拍暴力。

In [ ]:
def variable_elimination(factors, target, elim_order):
    # TODO: 见正文算法骨架；用 factor_product 与 factor_marginalize；最后归一化
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 用草地湿网络（前面已定义 p_R, p_S_R, p_W_RS, joint, brute_marginal）
facs = [p_R, p_S_R, p_W_RS]
pR_ve = variable_elimination(facs, 'R', ['S','W'])
pR_brute = brute_marginal(joint, 'R')
assert np.allclose(pR_ve.table, pR_brute.table, atol=1e-10), 'VE p(R) 应=暴力'
assert abs(pR_ve.table.sum() - 1.0) < 1e-12, '应归一化'
# p(W) 也对
pW_ve = variable_elimination(facs, 'W', ['R','S'])
assert np.allclose(pW_ve.table, brute_marginal(joint,'W').table, atol=1e-10)
print('✅ 练习 2 通过：变量消去 = 暴力边缘')

## ✏️ 练习 3：BP 因子→变量消息

信念传播的核心是 **因子→变量** 消息：$\mu_{f\to x}(x)=\sum_{\text{其他变量}} f \cdot\prod\text{传入消息}$。对一个二元因子 $f(x,y)$ 和传入的 $\mu_{y\to f}$，实现 `factor_to_var_message(f_table, incoming_y)` 返回 $\mu_{f\to x}(x)=\sum_y f(x,y)\mu_{y\to f}(y)$。

In [ ]:
def factor_to_var_message(f_table, incoming_y):
    # f_table: (kx, ky) 因子 f(x,y); incoming_y: (ky,) 来自 y 的消息
    # TODO: 返回 (kx,) 的消息 sum_y f(x,y)*incoming_y(y)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
f_table = np.array([[2.0, 1.0],[0.5, 3.0]])   # f(x,y), x,y∈{0,1}
incoming = np.array([1.0, 1.0])               # 均匀消息
msg = factor_to_var_message(f_table, incoming)
# 均匀消息下 = 对 y 求和: [2+1, 0.5+3]=[3, 3.5]
assert np.allclose(msg, [3.0, 3.5])
# 非均匀消息: y 偏向 1
msg2 = factor_to_var_message(f_table, np.array([0.0, 1.0]))
assert np.allclose(msg2, [1.0, 3.0]), '只保留 y=1 列'
# 形状正确
assert msg.shape == (2,)
print('✅ 练习 3 通过：因子→变量消息 = 乘传入消息再对其他变量求和')

## ✏️ 练习 4：d-分离的三种基本结构

不写完整算法，只判断三种基本结构的条件独立。实现 `is_active_triple(structure, observed)`：给定结构（'chain'/'fork'/'collider'）和中间节点是否被观测（bool），返回路径是否**活跃**（True=信息可通过=不独立）。

In [ ]:
def is_active_triple(structure, mid_observed):
    # chain X->Z->Y, fork X<-Z->Y: 观测Z则阻断(inactive); 否则活跃
    # collider X->Z<-Y: 观测Z则打开(active); 否则阻断(inactive)
    # TODO: 返回 True(活跃/不独立) 或 False(阻断/独立)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 链/叉: 观测中间 -> 阻断; 不观测 -> 活跃
assert is_active_triple('chain', mid_observed=False) == True
assert is_active_triple('chain', mid_observed=True) == False
assert is_active_triple('fork', mid_observed=False) == True
assert is_active_triple('fork', mid_observed=True) == False
# 对撞: 反过来! 观测中间 -> 打开; 不观测 -> 阻断
assert is_active_triple('collider', mid_observed=False) == False
assert is_active_triple('collider', mid_observed=True) == True
print('✅ 练习 4 通过：链/叉(观测则断) vs 对撞(观测则通) —— d-分离的核心')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def factor_marginalize(f, var):
    ax = f.vars.index(var)
    new_vars = tuple(v for v in f.vars if v != var)
    return Factor(new_vars, f.table.sum(axis=ax))

In [ ]:
# 练习 2 参考答案
def variable_elimination(factors, target, elim_order):
    Phi = list(factors)
    for z in elim_order:
        with_z = [f for f in Phi if z in f.vars]
        Phi = [f for f in Phi if z not in f.vars]
        if not with_z: continue
        prod = with_z[0]
        for f in with_z[1:]: prod = factor_product(prod, f)
        Phi.append(factor_marginalize(prod, z))
    result = Phi[0]
    for f in Phi[1:]: result = factor_product(result, f)
    result.table = result.table / result.table.sum()
    return result

In [ ]:
# 练习 3 参考答案
def factor_to_var_message(f_table, incoming_y):
    return (f_table * incoming_y[None, :]).sum(axis=1)

In [ ]:
# 练习 4 参考答案
def is_active_triple(structure, mid_observed):
    if structure in ('chain', 'fork'):
        return not mid_observed
    elif structure == 'collider':
        return mid_observed
    raise ValueError(structure)

---
## 🧪 真实数据胶囊：朴素贝叶斯垃圾邮件分类（PGM 的特例）

真实任务：**朴素贝叶斯**（Naive Bayes）是最简单的贝叶斯网——一个类节点 $C$ 指向所有特征 $w_i$，假设特征**给定类条件独立**（$p(C,w)=p(C)\prod_i p(w_i|C)$）。它就是一个星形贝叶斯网，推断 = 变量消去。

我们用一个真实风格的小型垃圾邮件词频数据（spam/ham 的典型词），训练朴素贝叶斯（数频率 + 模块 01 的 Dirichlet/Laplace 平滑），验证：① 分类合理；② 后验 $p(C|w)$ 正是贝叶斯网的条件推断。内置真实风格数据，无需联网。

In [ ]:
# 真实风格的垃圾邮件特征（词是否出现），标签 1=spam 0=ham
# 词表: ['free','money','meeting','project','win','report']
vocab = ['free','money','meeting','project','win','report']
# 训练数据: 每行一封邮件的词出现指示 + 标签（基于真实垃圾邮件常见模式构造）
X_train = np.array([
    [1,1,0,0,1,0],  # spam: free money win
    [1,1,0,0,0,0],  # spam: free money
    [1,0,0,0,1,0],  # spam: free win
    [0,1,0,0,1,0],  # spam: money win
    [1,1,0,0,1,0],  # spam
    [0,0,1,1,0,1],  # ham: meeting project report
    [0,0,1,0,0,1],  # ham: meeting report
    [0,0,0,1,0,1],  # ham: project report
    [0,0,1,1,0,0],  # ham: meeting project
    [0,0,1,1,0,1],  # ham
])
y_train = np.array([1,1,1,1,1, 0,0,0,0,0])

def train_naive_bayes(X, y, alpha=1.0):
    '''Laplace 平滑(=Dirichlet 先验, 模块01)。返回 p(C) 与 p(w_i=1|C)。'''
    classes = [0, 1]; n, d = X.shape
    p_c = np.array([(y==c).sum()+alpha for c in classes]); p_c/=p_c.sum()
    # p(w_i=1|C=c) = (含该词的c类样本数 + alpha)/(c类样本数 + 2*alpha)
    p_w_given_c = np.zeros((2, d))
    for c in classes:
        Xc = X[y==c]
        p_w_given_c[c] = (Xc.sum(axis=0) + alpha) / (len(Xc) + 2*alpha)
    return p_c, p_w_given_c

p_c, p_w_c = train_naive_bayes(X_train, y_train)

def nb_posterior(x, p_c, p_w_c):
    '''p(C|w) ∝ p(C) ∏ p(w_i|C)。在对数空间算（模块00）。'''
    log_post = np.log(p_c).copy()
    for c in [0,1]:
        pw = p_w_c[c]
        log_post[c] += np.sum(np.where(x==1, np.log(pw), np.log(1-pw)))
    log_post -= log_post.max()
    post = np.exp(log_post); return post/post.sum()

# 测试: 一封含 free+money 的邮件
test_spam = np.array([1,1,0,0,0,0])
post = nb_posterior(test_spam, p_c, p_w_c)
print(f"'free money' -> p(spam)={post[1]:.3f}")
assert post[1] > 0.8, '含垃圾词应判为 spam'
# 一封含 meeting+project 的邮件
test_ham = np.array([0,0,1,1,0,0])
post_ham = nb_posterior(test_ham, p_c, p_w_c)
print(f"'meeting project' -> p(spam)={post_ham[1]:.3f}")
assert post_ham[1] < 0.2, '含正常词应判为 ham'
# 训练集精度
preds = np.array([nb_posterior(x,p_c,p_w_c).argmax() for x in X_train])
acc = (preds==y_train).mean()
print(f'训练精度={acc:.2f}')
assert acc == 1.0, '这个可分数据上朴素贝叶斯应全对'
print('✅ 胶囊验证：朴素贝叶斯=星形贝叶斯网，后验推断=变量消去（特征条件独立）')

**🧪 胶囊练习**：实现 `nb_classify(x, p_c, p_w_c)`：用朴素贝叶斯后验给单封邮件分类（返回 0/1）。这就是在星形贝叶斯网上做 MAP 推断——本课所有概念（贝叶斯定理、条件独立、对数空间、图模型）的一次合奏。

In [ ]:
def nb_classify(x, p_c, p_w_c):
    # TODO: 算后验 p(C|w)（对数空间），返回 argmax 的类
    raise NotImplementedError

In [ ]:
# 自测
assert nb_classify(np.array([1,1,0,0,1,0]), p_c, p_w_c) == 1, 'free money win -> spam'
assert nb_classify(np.array([0,0,1,1,0,1]), p_c, p_w_c) == 0, 'meeting project report -> ham'
# 与后验 argmax 一致
for x in X_train:
    assert nb_classify(x,p_c,p_w_c) == nb_posterior(x,p_c,p_w_c).argmax()
print('✅ 胶囊练习通过：朴素贝叶斯分类 = 星形贝叶斯网上的 MAP 推断')

In [ ]:
# 📖 胶囊参考答案
def nb_classify(x, p_c, p_w_c):
    return int(nb_posterior(x, p_c, p_w_c).argmax())

### 小结
- **PGM** 用图表达高维联合分布，对抗维度灾难。核心三位一体：**图结构 ⟺ 条件独立 ⟺ 因子分解**。
- **贝叶斯网**（DAG）按 $\prod p(x_i|\mathrm{pa}_i)$ 分解；**MRF**（无向）按团势 $\frac1Z\prod\psi_c$ 分解（配分函数 $Z$ 是核心困难）。
- **d-分离** 纯拓扑读出条件独立：链/叉观测中间则阻断；**对撞观测则打开**（explaining away）。
- **变量消去** 用分配律聪明求和（含变量的因子相乘→消去）；复杂度由**树宽**（非变量数）决定。
- **信念传播 / sum-product** 在因子图上传消息，**树上精确**、一次前后向算出所有边缘；有环用 loopy BP（近似）。
- **精确（VE/BP/junction tree）vs 近似（MCMC/VI）由树宽分界**：低树宽用精确、高树宽转近似——串起模块 02/03。

**全课完结**：贝叶斯推断 → MCMC → 变分推断 → 高斯过程 → 概率图模型，五大支柱全部从零实现并对拍验证。你已掌握用概率视角思考、并亲手造出推断内核的能力。下一步：把这些内核映射到 PyMC/Stan/Pyro/GPyTorch，去解真实问题。